In [ ]:
# CIBC Digital Insurance Platform Acquisition
# Module 3 — Risk Scoring & Heat Map
# ALY6130 | Group 7 | June 2026
# -----------------------------------------
# Programmatically generates the risk heat map.
# Likelihood = X-axis | Impact = Y-axis
# Scales from ALY6130 Risk Calculation Sheet

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load the CIBC risk register
# Columns: Risk #, Risk Name, Likelihood, Impact, Risk Score, Priority, Type
df = pd.read_csv('master_risks.csv')
print('Risks loaded:', len(df))
print()
df.head(10)

In [ ]:
# Verify: Risk Score = Likelihood × Impact for every row
df['Calc_Score'] = df['Likelihood'] * df['Impact']
mismatches = df[df['Calc_Score'] != df['Risk Score']]
if mismatches.empty:
    print('All 30 scores verified: Risk Score = Likelihood × Impact ✓')
else:
    print('Mismatches found:')
    print(mismatches[['Risk #', 'Risk Name', 'Likelihood', 'Impact', 'Risk Score', 'Calc_Score']])

In [ ]:
# Classify severity using ALY6130 Risk Calculation Sheet thresholds
# HIGH >= 36  |  MEDIUM 10-35  |  LOW 1-9

def classify(score):
    if score >= 36:
        return 'High'
    elif score >= 10:
        return 'Medium'
    return 'Low'

df['Severity'] = df['Risk Score'].apply(classify)

print('Priority distribution:')
print(df['Severity'].value_counts().to_string())
print()
print(f"High  (Score >= 36): {len(df[df['Severity']=='High'])} risks")
print(f"Medium (Score 10-35): {len(df[df['Severity']=='Medium'])} risks")
print(f"Low   (Score 1-9):  {len(df[df['Severity']=='Low'])} risks")
print(f"Positive risks (opportunities): {len(df[df['Type']=='Positive'])}")

In [ ]:
# Threshold summary
summary = df.groupby('Severity').agg(
    Count=('Risk #', 'count'),
    Min_Score=('Risk Score', 'min'),
    Max_Score=('Risk Score', 'max'),
    Risk_IDs=('Risk #', lambda x: ', '.join('R' + str(i) for i in sorted(x)))
).reset_index()

print('Threshold Summary:')
print(summary.to_string(index=False))

In [ ]:
# All 30 risks ranked by Risk Score (highest first)
ranked = df.sort_values('Risk Score', ascending=False).reset_index(drop=True)
print('All 30 risks ranked by Risk Score:\n')
print(ranked[['Risk #', 'Risk Name', 'Likelihood', 'Impact', 'Risk Score', 'Priority', 'Type']]
      .to_string(index=False))

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# RISK HEAT MAP
# X-axis = Likelihood (scale: 1, 3, 5, 7, 9)
# Y-axis = Impact     (scale: 1, 2, 4, 6, 8, 9)
# Both scales from the ALY6130 Risk Calculation Sheet
# Risk Score = Likelihood × Impact
# Bubble size proportional to Risk Score
# Red = High (>=36) | Yellow = Medium (10-35) | Green = Low (1-9) | Blue = Positive
# ─────────────────────────────────────────────────────────────────────────────

# Group risks by (Likelihood, Impact) coordinate
groups = defaultdict(list)
for _, row in df.iterrows():
    groups[(row['Likelihood'], row['Impact'])].append(row)

# Jitter overlapping risks at the same coordinate
def jitter(n):
    if n == 1: return [(0, 0)]
    if n == 2: return [(-0.22, 0), (0.22, 0)]
    if n == 3: return [(-0.24, 0.14), (0.24, 0.14), (0, -0.20)]
    if n == 4: return [(-0.22, 0.17), (0.22, 0.17), (-0.22, -0.17), (0.22, -0.17)]
    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)
    return [(0.27 * np.cos(a), 0.27 * np.sin(a)) for a in angles]

# Colour by threshold and type
def get_colours(score, risk_type):
    if risk_type == 'Positive':  return '#4472C4', '#1F3864', 'white'
    if score >= 36:              return '#D42020', '#8B0000', 'white'
    if score >= 10:              return '#FFB300', '#7A5500', '#1A0A00'
    return '#3DAA2A', '#1E5018', 'white'

# Create figure
fig, ax = plt.subplots(figsize=(16, 11))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

# Draw background threshold zones (score = L × I)
xs = np.linspace(0.5, 9.5, 400)
ys = np.linspace(0.5, 9.5, 400)
XX, YY = np.meshgrid(xs, ys)
ZZ = XX * YY
ax.contourf(XX, YY, ZZ, levels=[0, 10, 36, 82],
            colors=['#D5F0CC', '#FFF3C4', '#FFD0D0'], alpha=0.45, zorder=0)
ax.contour(XX, YY, ZZ, levels=[10, 36],
           colors=['#DAA500', '#CC0000'], linewidths=[1.3, 2.0],
           linestyles=['--', '-'], zorder=1)

# Zone labels
ax.text(1.3, 1.3, 'LOW\n(1-9)',     ha='center', va='center', fontsize=10,
        fontweight='bold', color='#2E7D32', alpha=0.45, zorder=2)
ax.text(3.5, 3.2, 'MEDIUM\n(10-35)', ha='center', va='center', fontsize=10,
        fontweight='bold', color='#7A5500', alpha=0.45, zorder=2)
ax.text(7.5, 7.5, 'HIGH\n(≥36)',    ha='center', va='center', fontsize=11,
        fontweight='bold', color='#8B0000', alpha=0.45, zorder=2)

# Plot each risk as a bubble
for (L, I), group in groups.items():
    offsets = jitter(len(group))
    for (dx, dy), risk in zip(offsets, group):
        score = risk['Risk Score']
        face, edge, text_col = get_colours(score, risk['Type'])
        ax.scatter(L + dx, I + dy,
                   s=260 + score * 20,
                   c=face, edgecolors=edge,
                   linewidths=1.8, zorder=4, alpha=0.93)
        ax.text(L + dx, I + dy, str(int(risk['Risk #'])),
                ha='center', va='center',
                fontsize=8.5, fontweight='bold', color=text_col, zorder=5)

# Axes — exact values from Risk Calculation Sheet only
ax.set_xticks([1, 3, 5, 7, 9])
ax.set_xticklabels(
    ['1\n(Very Unlikely)', '3\n(Somewhat Unlikely)',
     '5\n(50-50)', '7\n(Somewhat Likely)', '9\n(Very Likely)'],
    fontsize=9, fontweight='bold')
ax.set_yticks([1, 2, 4, 6, 8, 9])
ax.set_yticklabels(
    ['1\n(Very Low)', '2\n(Somewhat Low)', '4\n(Moderate)',
     '6\n(Somewhat High)', '8\n(High)', '9\n(Extremely High)'],
    fontsize=9, fontweight='bold')
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.set_xlabel('LIKELIHOOD  (X-axis)', fontsize=11, fontweight='bold', labelpad=12)
ax.set_ylabel('IMPACT  (Y-axis)', fontsize=11, fontweight='bold', labelpad=12)
ax.set_title(
    'CIBC — Digital Insurance Platform Acquisition\n'
    'Risk Heat Map  |  Likelihood (X-axis) × Impact (Y-axis)  |  '
    'ALY6130  |  Group 7  |  June 2026',
    fontsize=13, fontweight='bold', pad=16)

# Legend
legend_patches = [
    mpatches.Patch(facecolor='#D42020', edgecolor='#8B0000', label='HIGH — Risk Score ≥ 36'),
    mpatches.Patch(facecolor='#FFB300', edgecolor='#7A5500', label='MEDIUM — Risk Score 10–35'),
    mpatches.Patch(facecolor='#3DAA2A', edgecolor='#1E5018', label='LOW — Risk Score 1–9'),
    mpatches.Patch(facecolor='#4472C4', edgecolor='#1F3864', label='POSITIVE RISK — Opportunity'),
    mpatches.Patch(facecolor='none',    edgecolor='none',    label='Bubble size ∝ Risk Score  |  Number = Risk #'),
]
legend = ax.legend(handles=legend_patches, loc='lower right', fontsize=9,
                   framealpha=0.95, edgecolor='#CCCCCC',
                   title='THRESHOLD LEGEND', title_fontsize=10)
legend.get_title().set_fontweight('bold')

# Figure caption
fig.subplots_adjust(bottom=0.10)
fig.text(0.04, 0.02,
    'Figure 3. Python-generated risk heat map. X-axis = Likelihood (scale: 1,3,5,7,9); '
    'Y-axis = Impact (scale: 1,2,4,6,8,9). '
    'Risk Score = Likelihood × Impact. '
    'Source: Group 7 Risk Register and Risk Calculation Sheet, ALY6130, Spring 2026.',
    fontsize=7, color='#777777', style='italic')

plt.savefig('fig1_python_heatmap.png', dpi=160, bbox_inches='tight', facecolor='white')
plt.show()
print('Heat map saved as fig1_python_heatmap.png')